# Classes and Methods: Build a Banking Application

We will use one small banking application as a continuing problem statement. Starting from a single account, we will add methods, protect account state, model account types with inheritance, support polymorphic month-end behavior, and finish with an abstract account design.

> **Learning goal:** Understand the ideas by improving the same program one step at a time. This is a teaching example, not production banking software.

## Visual introduction

The bank-app illustration gives us the domain: accounts, balances, deposits, withdrawals, and a protected way to access money.

![Banking application illustration](Images/OOPS_template_bank.jpg)

The companion OOP notebook already links to this class-and-object diagram. It is reused here to show the relationship between a class and its instances.

![Class and object diagram from the companion OOP notebook](https://i.imgur.com/c82ZQmfl.png "Class and object")

Here is a real-world ATM photo found through Wikimedia Commons. The file is marked CC0; it provides context, while the animated diagram below focuses on how our software objects respond to operations.

![ATMs in Croatia, by Gewild, CC0](https://upload.wikimedia.org/wikipedia/commons/thumb/3/31/ATM-Croatia-2024.jpg/960px-ATM-Croatia-2024.jpg)

Source: [ATMs in Croatia on Wikimedia Commons](https://commons.wikimedia.org/wiki/File:ATM-Croatia-2024.jpg) — Gewild, CC0 1.0.

The animation was generated with Python's Matplotlib animation tools. Follow the customer request into the account object, then see the balance and ledger change after each operation.

![Animated banking account transaction flow](Images/banking_transaction_flow.gif)

## The problem statement

Create a small in-memory banking application that can:

1. Create customer accounts with an opening balance.
2. Deposit money and withdraw money.
3. Reject invalid amounts and withdrawals that exceed the permitted funds.
4. Transfer money between accounts.
5. Keep a simple transaction history.
6. Support savings and checking accounts with different rules.
7. Apply month-end behavior without checking each account type at every call site.

We will use `Decimal` for money instead of binary floating-point values such as `0.1`. The examples use USD for readability; the currency can be changed for the exercise.

## Learning path

| Stage | OOP idea | Banking example |
|---|---|---|
| 1 | Classes and objects | A `BankAccount` blueprint and individual accounts |
| 2 | Methods | `deposit`, `withdraw`, `transfer_to`, and month-end actions |
| 3 | Encapsulation | Keep the balance behind validated operations |
| 4 | Inheritance | Add `SavingsAccount` and `CheckingAccount` |
| 5 | Polymorphism | Call the same `monthly_update` method on different accounts |
| 6 | Abstraction | Define the common rules in an abstract `BankAccount` |

## 1. Classes and objects: describe an account

A **class** describes the data and behavior that its objects share. An **object** is one particular instance of that class, with its own account number, owner, and balance. The class is the blueprint; `asha_account` and `lee_account` are separate accounts created from it.

At first, keep the class small. The `__init__` method initializes each new object, and `self` refers to the particular object receiving the method call.

In [1]:
class BankAccount:

    def __init__(self, owner, initial_balance):
        self.owner = owner
        self.initial_balance = initial_balance

        

In [2]:
AmanSah = BankAccount(owner= "Aman", initial_balance= 5000)

In [4]:
AmanSah.initial_balance

5000

In [7]:
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance


asha_account = BankAccount("Asha", 500)
lee_account = BankAccount("Lee", 250)

print(asha_account.owner, asha_account.balance)
print(lee_account.owner, lee_account.balance)

Asha 500
Lee 250


In [8]:
aaksh = BankAccount("Aakash", 5444)
print(aaksh.balance)

5444


In [ ]:
BankAccount.deposit()

The two objects share the same class definition but keep independent state. A class is useful here because account data and account behavior belong together.

## 2. Methods: give an account behavior

A **method** is a function defined on a class. Instance methods receive `self`, so they can read or change the state of the object they were called on. A deposit should change one account, not a global balance shared by every customer.

The companion notebook also uses a fields-and-methods illustration. Fields represent state; methods represent actions that operate on that state.

![Fields and methods diagram reused from the companion OOP notebook](https://i.imgur.com/bpI6Kswl.jpg "Fields and methods")

Add operations to the beginner account:

In [9]:
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance


    #Method 1 : deposit 
    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Deposit must be positive")
        else : 
            self.balance += amount
    # Method 2 
    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError("Withdrawal must be positive")
        if amount > self.balance:
            raise ValueError("Insufficient funds")
        self.balance -= amount




In [15]:
account.balance

1300

In [12]:
account = BankAccount("Asha", 500)
account.deposit(2000)

In [14]:
account.withdraw(1200)

In [16]:
account.balance 

1300

In [18]:
account.balance = -100
account.balance

-100

This version is intentionally simple. In the next section, we will stop allowing callers to edit `balance` directly.

Python also supports class methods and static methods. An instance method works with one account; a class method works with class-level information and receives `cls`; a static method is a utility grouped with the class but receives neither `self` nor `cls`.

In [ ]:
class AccountRules:
    CURRENCY = "USD"

    @classmethod
    def currency_code(cls):
        return cls.CURRENCY

    @staticmethod
    def is_positive_amount(amount):
        return amount > 0


print(AccountRules.currency_code())
print(AccountRules.is_positive_amount(25))

## 3. Encapsulation: protect the balance

**Encapsulation** groups state with the methods that manage it and limits unsafe direct changes. For example, changing `account.balance` directly could create a negative balance without recording a withdrawal. Instead, expose a read-only property and update the balance only through validated methods.

Python's double underscore triggers name mangling; it is not a security boundary. A leading single underscore is the conventional signal that an attribute is an internal implementation detail. The important design rule is that callers use the public methods rather than modifying internal state.

![Encapsulation: state is managed through methods](Images/OOPS_encapsulations.jpg)

In [24]:
class BankAccount:
    def __init__(self, owner, opening_balance):
        if opening_balance < 0:
            raise ValueError("Opening balance cannot be negative")
        self.owner = owner
        self.__balance = opening_balance

    @property
    def balance(self):
        return self.__balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Deposit must be positive")
        self.__balance += amount


account = BankAccount("Asha", 500)
account.deposit(50)
print(account.balance)#   # 550
# account.balance = 0  # AttributeError: the property has no setter

550


The property gives callers safe read access, while the deposit method controls how state changes. In the final version, withdrawals and transfers follow the same rule and each successful operation is recorded.

## 4. Inheritance: specialize account rules

**Inheritance** lets a more specific class reuse and extend a general class. A savings account and a checking account are both bank accounts, so they can share owner, balance, deposit, and transaction behavior while adding different withdrawal rules.

The local diagram introduces a parent and child class; the companion OOP notebook also links to an inheritance schematic.

![Parent and child classes](Images/Inheritance.jpg)

![Inheritance schematic reused from the companion OOP notebook](https://i.imgur.com/DF6P030l.jpg "Inheritance")

`super().__init__(...)` runs the parent's initialization so the child does not duplicate common setup.

In [26]:
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance
    

class SavingsAccount(BankAccount):
    def __init__(self, owner, balance=0, minimum_balance=100):
        super().__init__(owner, balance)
        self.minimum_balance = minimum_balance

    def withdraw(self, amount):
        if self.balance - amount < self.minimum_balance:
            raise ValueError("Minimum balance would be breached")
        self.balance -= amount


savings = SavingsAccount("Asha", 1000, minimum_balance=100)
savings.withdraw(50)
print(savings.balance)  

950


The short example shows the relationship, but it repeats validation and exposes `balance`. The integrated example later keeps shared rules in the base class and encapsulates the actual balance.

## 5. Polymorphism: one call, account-specific behavior

**Polymorphism** means code can use a shared operation while different objects provide the behavior appropriate to their type. Both savings and checking accounts can respond to `monthly_update()`, even though only a savings account adds interest.

The local diagram shows different shapes responding through a shared idea; here, the shared operation is a month-end update.

![Polymorphism with different implementations of a shared operation](Images/Polymorphism.jpg)

In [ ]:
class SavingsAccount:
    def monthly_update(self):
        print("Add interest to savings")


class CheckingAccount:
    def monthly_update(self):
        print("Apply checking account fees, if any")


accounts = [SavingsAccount(), CheckingAccount()]
for account in accounts:
    account.monthly_update()

The loop does not need an `if` statement to determine the account type. Each object handles the operation through its own method implementation.

## 6. Abstraction: require a common account contract

**Abstraction** exposes the operations that callers need and hides details they do not need. A customer can ask an account to withdraw money without knowing whether that account enforces a minimum balance or an overdraft limit.

An abstract base class describes required behavior. It cannot be instantiated while it has abstract methods; concrete child classes must implement those methods before they can be used.

![Abstraction: a simplified view of information and behavior](Images/Abstraction.jpg)

## The abc library in Python stands for Abstract Base Classes.
When you import from abc import ABC, abstractmethod, you are bringing in the tools needed to define blueprints for other classes. It allows you to practice Object-Oriented Programming (OOP) strictly by enforcing common structures across different objects.
Here is exactly what these two imported items do:

1. ABC (The Blueprint Parent)

ABC is a helper class. When you make a new class inherit from ABC, you are telling Python: "This is an abstract class. It is just a concept/blueprint and cannot be created (instantiated) directly."

2. @abstractmethod (The Rules)

This is a decorator you place above methods inside your abstract class. It acts as a strict rule that says: "Any child class that inherits from this blueprint MUST write its own version of this method. If it doesn't, Python will throw an error."

In [27]:
from abc import ABC, abstractmethod

# 1. This is the blueprint. You cannot do: my_vehicle = Vehicle()
class Vehicle(ABC):
    
    @abstractmethod 
    def start_engine(self):
        """Every vehicle MUST implement this method!"""
        pass


In [ ]:
from abc import ABC, abstractmethod


class BankAccount(ABC):
    @abstractmethod
    def withdraw(self, amount):
        """Withdraw money according to this account's rules."""
        raise NotImplementedError


class SavingsAccount(BankAccount):
    def withdraw(self, amount):
        print(f"Check the savings minimum before withdrawing {amount}")

This contract makes it explicit that every usable account type must define withdrawal behavior. The complete example also requires each account type to define its month-end update.

## 7. Advanced example: a small banking application

This implementation combines the ideas:

- **Classes and methods:** accounts expose `deposit`, `withdraw`, and `transfer_to`.
- **Encapsulation:** the balance is private and has a read-only property.
- **Inheritance:** savings and checking accounts reuse a common base class.
- **Polymorphism:** `monthly_update` behaves differently for each concrete account.
- **Abstraction:** the abstract base class requires `withdraw` and `monthly_update`.
- **Safer money arithmetic:** amounts are `Decimal` values constructed from strings.

Read the contract and validation first: amounts must be positive `Decimal` values, and each account type controls its own withdrawal limit. Then follow how a successful operation updates the balance and appends a transaction.

In [ ]:
from __future__ import annotations

from abc import ABC, abstractmethod
from dataclasses import dataclass
from decimal import Decimal
from typing import Iterable


class InsufficientFundsError(ValueError):
    """Raised when an account cannot cover a withdrawal."""



""" 
What this class does automatically

Without the @dataclass decorator, you would have to manually write out a lot of Python code to handle initialization and string formatting. The decorator automatically generates:
• An __init__ constructor: You can instantly create objects using Transaction(kind="deposit", amount=Decimal("100.00"), balance_after=Decimal("500.00")).
• A readable __repr__ output: If you print the object, it looks clean and readable: Transaction(kind='deposit', amount=Decimal('100.00'), balance_after=Decimal('500.00')) (instead of a messy memory address).
• Comparison methods (__eq__): You can easily check if two separate transaction objects share the exact same data using ==.




"""

@dataclass(frozen=True)
class Transaction:
    kind: str
    amount: Decimal
    balance_after: Decimal


class BankAccount(ABC):
    CURRENCY = "USD"

    def __init__(
        self,
        account_number: str,
        owner: str,
        opening_balance: Decimal = Decimal("0.00"),
    ) -> None:
        if not account_number.strip():
            raise ValueError("Account number is required")
        if not owner.strip():
            raise ValueError("Owner name is required")
        if not isinstance(opening_balance, Decimal):
            raise TypeError("Use Decimal values for money")
        if opening_balance < 0:
            raise ValueError("Opening balance cannot be negative")

        self.__account_number = account_number
        self.__owner = owner
        self.__balance = Decimal("0.00")
        self.__transactions: list[Transaction] = []
        if opening_balance > 0:
            self.deposit(opening_balance)

    @property
    def account_number(self) -> str:
        return self.__account_number

    @property
    def owner(self) -> str:
        return self.__owner

    @property
    def balance(self) -> Decimal:
        return self.__balance

    @property
    def transactions(self) -> tuple[Transaction, ...]:
        return tuple(self.__transactions)

    @classmethod
    def currency_code(cls) -> str:
        return cls.CURRENCY

    @staticmethod
    def _validate_amount(amount: Decimal) -> None:
        if not isinstance(amount, Decimal):
            raise TypeError("Use Decimal values for money")
        if amount <= 0:
            raise ValueError("Amount must be positive")

    def _record(self, kind: str, amount: Decimal) -> None:
        self.__transactions.append(
            Transaction(kind, amount, self.__balance)
        )

    def deposit(self, amount: Decimal) -> None:
        self._validate_amount(amount)
        self.__balance += amount
        self._record("deposit", amount)

    def _debit(self, amount: Decimal, kind: str = "withdrawal") -> None:
        self._validate_amount(amount)
        self.__balance -= amount
        self._record(kind, amount)

    def transfer_to(self, destination: BankAccount, amount: Decimal) -> None:
        if not isinstance(destination, BankAccount):
            raise TypeError("Destination must be a bank account")
        if destination is self:
            raise ValueError("Cannot transfer to the same account")
        self._validate_amount(amount)
        self.withdraw(amount)
        destination.deposit(amount)

    @abstractmethod
    def withdraw(self, amount: Decimal) -> None:
        """Withdraw according to the concrete account's rules."""

    @abstractmethod
    def monthly_update(self) -> Decimal:
        """Apply the account type's month-end behavior."""


class SavingsAccount(BankAccount):
    def __init__(
        self,
        account_number: str,
        owner: str,
        opening_balance: Decimal = Decimal("0.00"),
        minimum_balance: Decimal = Decimal("100.00"),
        monthly_interest_rate: Decimal = Decimal("0.01"),
    ) -> None:
        if not isinstance(minimum_balance, Decimal) or minimum_balance < 0:
            raise ValueError("Minimum balance must be a non-negative Decimal")
        if (
            not isinstance(monthly_interest_rate, Decimal)
            or monthly_interest_rate < 0
        ):
            raise ValueError("Interest rate must be a non-negative Decimal")
        self._minimum_balance = minimum_balance
        self._monthly_interest_rate = monthly_interest_rate
        super().__init__(account_number, owner, opening_balance)

    def withdraw(self, amount: Decimal) -> None:
        self._validate_amount(amount)
        if self.balance - amount < self._minimum_balance:
            raise InsufficientFundsError("Savings minimum balance would be breached")
        self._debit(amount)

    def monthly_update(self) -> Decimal:
        interest = (self.balance * self._monthly_interest_rate).quantize(
            Decimal("0.01")
        )
        if interest > 0:
            self.deposit(interest)
        return interest


class CheckingAccount(BankAccount):
    def __init__(
        self,
        account_number: str,
        owner: str,
        opening_balance: Decimal = Decimal("0.00"),
        overdraft_limit: Decimal = Decimal("0.00"),
    ) -> None:
        if not isinstance(overdraft_limit, Decimal) or overdraft_limit < 0:
            raise ValueError("Overdraft limit must be a non-negative Decimal")
        self._overdraft_limit = overdraft_limit
        super().__init__(account_number, owner, opening_balance)

    def withdraw(self, amount: Decimal) -> None:
        self._validate_amount(amount)
        if amount > self.balance + self._overdraft_limit:
            raise InsufficientFundsError("Overdraft limit would be exceeded")
        self._debit(amount)

    def monthly_update(self) -> Decimal:
        return Decimal("0.00")


def apply_month_end(accounts: Iterable[BankAccount]) -> None:
    for account in accounts:
        change = account.monthly_update()
        print(
            f"{account.account_number}: "
            f"month-end change {change} {account.currency_code()}"
        )


savings = SavingsAccount(
    "SA-1001",
    "Asha",
    Decimal("1000.00"),
    monthly_interest_rate=Decimal("0.01"),
)
checking = CheckingAccount(
    "CA-2001",
    "Lee",
    Decimal("300.00"),
    overdraft_limit=Decimal("100.00"),
)

checking.deposit(Decimal("50.00"))
checking.withdraw(Decimal("25.00"))
savings.transfer_to(checking, Decimal("100.00"))

apply_month_end([savings, checking])

for account in (savings, checking):
    print(
        account.account_number,
        account.owner,
        account.balance,
        account.currency_code(),
    )

for transaction in savings.transactions:
    print(transaction.kind, transaction.amount, transaction.balance_after)

Expected final balances are `SA-1001: 909.00` and `CA-2001: 425.00`. The savings account receives `9.00` interest after transferring `100.00`; the checking account does not earn interest in this example.

## What this example deliberately does not solve

This code is an OOP teaching model, not a real financial system. In particular, a transfer here performs a withdrawal and then a deposit in memory; it is not an atomic database transaction. A production system needs persistence, concurrency control, authentication and authorization, audit requirements, idempotency, currency rules, and carefully tested failure recovery. Do not store or process real customer credentials or money with this example.

## Practice tasks

1. Add a `withdraw` operation to the beginner class and reject zero or negative amounts.
2. Add a `minimum_balance` rule to savings and an `overdraft_limit` rule to checking.
3. Add a `BusinessAccount` subclass with a different month-end rule.
4. Add a method that prints a statement from the transaction history.
5. Write tests for exact-balance withdrawal, insufficient funds, invalid amounts, transfers, and month-end behavior.
6. Explain which methods belong in the abstract base class and which must remain specific to each account type.

## References

- [Python `decimal` documentation](https://docs.python.org/3/library/decimal.html)
- [Python abstract base classes (`abc`)](https://docs.python.org/3/library/abc.html)
- [ATMs in Croatia image source and license](https://commons.wikimedia.org/wiki/File:ATM-Croatia-2024.jpg)
- The class/object, fields/methods, and inheritance image links above are reused from `03_1b_Object_Oriented_Python.ipynb`.